# EM rollout flux comparison: GKW vs gyaradax

Visual comparison notebook for the structured EM validation tree. The layout follows the validation structure rather than pass/fail status.

Use `STAGE = 'rollout_full'` for full comparisons or `STAGE = 'rollout_short'` for quick visual checks. Data are read from `/local00/bioinf/volkmann/gyrokinetics/em_validation/<regime>/{gkw,gyaradax}/<stage>/<case>`.

Provenance notes from `gkw_ref`:

- `bpar_bench_*`: from `gkw_ref/benchmarks/bpar`; the README says these inputs are Waltz-test-case based and benchmark compressional magnetic-field (`B_parallel`) support.
- `beta_bench_*`: from `gkw_ref/benchmarks/beta`; the README identifies these as the beta-scan inputs from the GKW CPC paper, with matching GS2 inputs.
- `waltz_bpar_linear`: from `gkw_ref/tests/standard/bpar_waltz_linear`; a linear Waltz A_parallel+B_parallel case with open parallel boundary conditions, nonzero `uprim`, and `VCOR = 0.1`. Current gyaradax does **not** support the `VCOR`/rotation physics, so this is a useful setting/provenance comparison, not an apples-to-apples strict physics match.
- `diag_lin_all_em`: adapted from `gkw_ref/tests/extra/diag_lin_all`; broad linear diagnostic coverage with A_parallel+B_parallel, collisions, `uprim`, `VCOR`, and centrifugal flags. Current gyaradax does **not** support `VCOR`/rotation physics.
- `linear_*` and `nonlinear_*` Waltz cases are generated validation templates for controlled field-set comparisons (`A_parallel`, `B_parallel`, or both) at selected beta values.


In [ ]:
from __future__ import annotations

from pathlib import Path
from io import StringIO
import json
import re

import numpy as np
try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None
import matplotlib.pyplot as plt

DATA_ROOT = Path('/local00/bioinf/volkmann/gyrokinetics/em_validation')
REGISTRY_PATH = Path('../gkw_ref/em_validation_cases/index.json')
if not REGISTRY_PATH.exists():
    REGISTRY_PATH = Path('gkw_ref/em_validation_cases/index.json')

STAGE = 'rollout_full'  # or 'rollout_short'

DATASETS = {
    'fluxes': ('fluxes.npy', 'fluxes.dat', ['ion p', 'ion e', 'ion v', 'elec p', 'elec e', 'elec v']),
    'fluxes_em': ('fluxes_em.npy', 'fluxes_em.dat', ['ion p', 'ion e', 'ion v', 'elec p', 'elec e', 'elec v']),
    'fluxes_bpar': ('fluxes_bpar.npy', 'fluxes_bpar.dat', ['ion p', 'ion e', 'ion v', 'elec p', 'elec e', 'elec v']),
}

with REGISTRY_PATH.open(encoding='utf-8') as f:
    REGISTRY = json.load(f)
CASES = [case for case in REGISTRY['cases'] if case['stage'] == STAGE]

def cases_for(regime: str, setting: str | None = None) -> list[dict]:
    rows = [case for case in CASES if case['regime'] == regime]
    if setting is not None:
        rows = [case for case in rows if case['setting'] == setting]
    return sorted(rows, key=lambda c: c['case'])

def load_gkw_table(path: Path) -> np.ndarray | None:
    if not path.exists():
        return None
    raw_lines = [line for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not raw_lines:
        return None
    text = '\n'.join(raw_lines)
    text = re.sub(r'(?<=[0-9.])([+-][0-9]{2,})(?=\s|$)', r'E\1', text)
    arr = np.asarray(np.loadtxt(StringIO(text)))
    if arr.ndim == 0:
        return arr.reshape(1, 1)
    if arr.ndim == 1:
        return arr.reshape(1, -1) if len(raw_lines) == 1 else arr.reshape(-1, 1)
    return arr

def case_dirs(case: dict) -> tuple[Path, Path]:
    structured = case['paths']['structured']
    return Path(structured['gkw']), Path(structured['gyaradax'])

def load_case(case: dict):
    gkw_dir, gy_dir = case_dirs(case)
    if not gy_dir.exists():
        raise FileNotFoundError(f'Missing gyaradax dir: {gy_dir}')
    if not gkw_dir.exists():
        raise FileNotFoundError(f'Missing GKW dir: {gkw_dir}')
    gy_time = np.load(gy_dir / 'time.npy').reshape(-1)
    gkw_time_table = load_gkw_table(gkw_dir / 'time.dat')
    if gkw_time_table is None:
        raise FileNotFoundError(f'Missing or empty GKW time.dat: {gkw_dir / 'time.dat'}')
    return gkw_dir, gy_dir, gy_time, gkw_time_table[:, 0]

def common_time_grid(gy_time: np.ndarray, gkw_time: np.ndarray) -> np.ndarray:
    start = max(float(np.nanmin(gy_time)), float(np.nanmin(gkw_time)))
    end = min(float(np.nanmax(gy_time)), float(np.nanmax(gkw_time)))
    return gkw_time[(gkw_time >= start) & (gkw_time <= end)]

def interp_columns(src_t: np.ndarray, values: np.ndarray, dst_t: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    if values.ndim == 1:
        values = values.reshape(-1, 1)
    return np.stack([np.interp(dst_t, src_t, values[:, i]) for i in range(values.shape[1])], axis=1)

def aligned_dataset(case: dict, dataset: str):
    gkw_dir, gy_dir, gy_time, gkw_time = load_case(case)
    npy_name, dat_name, labels = DATASETS[dataset]
    gy_path = gy_dir / npy_name
    gkw_path = gkw_dir / dat_name
    if not gy_path.exists() or not gkw_path.exists():
        return None
    gy = np.load(gy_path)
    gkw = load_gkw_table(gkw_path)
    if gkw is None:
        return None
    t = common_time_grid(gy_time, gkw_time)
    gkw_mask = np.isin(gkw_time, t)
    gy_i = interp_columns(gy_time, gy, t)
    gkw_i = gkw[gkw_mask, :gy_i.shape[1]]
    ncols = min(gy_i.shape[1], gkw_i.shape[1])
    return t, gy_i[:, :ncols], gkw_i[:, :ncols], labels[:ncols]

def metrics(gy: np.ndarray, gkw: np.ndarray, floor: float = 1e-30) -> dict:
    denom = np.maximum(np.abs(gkw), floor)
    rel = np.abs(gy - gkw) / denom
    log_corrs = []
    for col in range(gy.shape[1]):
        mask = (np.abs(gy[:, col]) > floor) & (np.abs(gkw[:, col]) > floor)
        if mask.sum() >= 3:
            x = np.log(np.abs(gy[mask, col]))
            y = np.log(np.abs(gkw[mask, col]))
            if np.std(x) > 0 and np.std(y) > 0:
                log_corrs.append(np.corrcoef(x, y)[0, 1])
    sign_mask = (np.abs(gy) > floor) & (np.abs(gkw) > floor)
    return {'max_gkw': float(np.nanmax(np.abs(gkw))), 'max_gyaradax': float(np.nanmax(np.abs(gy))), 'median_rel': float(np.nanmedian(rel)), 'max_rel': float(np.nanmax(rel)), 'log_corr_median': float(np.nanmedian(log_corrs)) if log_corrs else np.nan, 'sign_agreement': float(np.mean(np.sign(gy[sign_mask]) == np.sign(gkw[sign_mask]))) if sign_mask.any() else np.nan}

print(f'Loaded {len(CASES)} {STAGE} cases from {REGISTRY_PATH}')


## Summary table


In [ ]:
rows = []
for case in CASES:
    for dataset in DATASETS:
        aligned = aligned_dataset(case, dataset)
        if aligned is None:
            continue
        t, gy, gkw, labels = aligned
        rows.append({'regime': case['regime'], 'setting': case['setting'], 'case': case['case'], 'dataset': dataset, 'points': len(t), 't_start': t[0], 't_end': t[-1], **metrics(gy, gkw)})
if pd is not None:
    summary = pd.DataFrame(rows)
    display(summary.sort_values(['regime', 'setting', 'case', 'dataset']))
else:
    summary = rows
    for row in rows:
        print(row)


## Plot helpers


In [ ]:
def case_title(case: dict) -> str:
    return f"{case['case']} ({case['field_set']}, beta={case['physics'].get('beta')})"

def plot_case(case: dict, datasets=('fluxes', 'fluxes_em', 'fluxes_bpar')):
    for dataset in datasets:
        aligned = aligned_dataset(case, dataset)
        if aligned is None:
            continue
        t, gy, gkw, labels = aligned
        ncols = gy.shape[1]
        fig, axes = plt.subplots(ncols, 1, figsize=(12, 2.2 * ncols), sharex=True)
        axes = np.atleast_1d(axes)
        for i, ax in enumerate(axes):
            ax.plot(t, gkw[:, i], label='GKW', lw=1.6)
            ax.plot(t, gy[:, i], label='gyaradax', lw=1.2, ls='--')
            ax.set_ylabel(labels[i])
            ax.grid(True, alpha=0.25)
            if i == 0:
                ax.set_title(f"{case_title(case)}: {dataset}")
                ax.legend(loc='best')
        axes[-1].set_xlabel('physical time')
        plt.tight_layout()
        plt.show()

def plot_case_abs(case: dict, datasets=('fluxes', 'fluxes_em', 'fluxes_bpar')):
    for dataset in datasets:
        aligned = aligned_dataset(case, dataset)
        if aligned is None:
            continue
        t, gy, gkw, labels = aligned
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(t, np.max(np.abs(gkw), axis=1), label='GKW max |columns|', lw=1.8)
        ax.plot(t, np.max(np.abs(gy), axis=1), label='gyaradax max |columns|', lw=1.4, ls='--')
        ax.set_yscale('log')
        ax.set_title(f"{case_title(case)}: {dataset} max absolute flux")
        ax.set_xlabel('physical time')
        ax.set_ylabel('max |flux column|')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()

def plot_group(regime: str, setting: str, *, datasets=('fluxes', 'fluxes_em', 'fluxes_bpar'), max_cases: int | None = None):
    group = cases_for(regime, setting)
    if max_cases is not None:
        group = group[:max_cases]
    print(f'{regime}/{setting}: {len(group)} case(s) for stage={STAGE}')
    for case in group:
        plot_case_abs(case, datasets=datasets)
        plot_case(case, datasets=datasets)


## Linear

Linear EM validation cases grouped by tested setting. These sections include published/manual benchmark-derived scans, GKW standard/extra regression settings, and generated controlled Waltz variants.


### bpar_waltz_apar_beta_scan

Bpar benchmark family from `gkw_ref/benchmarks/bpar`, but with `B_parallel` disabled (`nlapar=true`, `nlbpar=false`). The benchmark README says the inputs are based on the Waltz test case and were used for compressional magnetic-field benchmark work. This subgroup isolates the A_parallel branch over the selected beta scan.


In [ ]:
plot_group('linear', 'bpar_waltz_apar_beta_scan')


### bpar_waltz_apar_bpar_beta_scan

Bpar benchmark family from `gkw_ref/benchmarks/bpar` with both `A_parallel` and compressional `B_parallel` enabled. This is the Waltz-based finite-beta compressional magnetic-field benchmark axis.


In [ ]:
plot_group('linear', 'bpar_waltz_apar_bpar_beta_scan')


### cpc_apar_beta_scan

CPC-paper beta scan from `gkw_ref/benchmarks/beta`. The README states these are the beta-scan input files from the CPC paper, with both GKW and GS2 inputs. In this registry this is the finite-beta A_parallel benchmark family.


In [ ]:
plot_group('linear', 'cpc_apar_beta_scan')


### bpar_waltz_open_parallel_boundary

GKW standard `bpar_waltz_linear` setting. It is a linear Waltz A_parallel+B_parallel case using `parallel_boundary_conditions = \"open\"`. The original input also has nonzero `uprim` and `VCOR = 0.1`; current gyaradax does **not** support `VCOR`/rotation physics, so interpret this as an open-boundary/provenance comparison with known extra GKW physics.


In [ ]:
plot_group('linear', 'bpar_waltz_open_parallel_boundary')


### linear_apar_waltz

Generated controlled linear Waltz-style cases with A_parallel enabled and B_parallel disabled, at selected beta values. These are not directly documented as a separate GKW benchmark family; they are generated validation settings for isolating the A_parallel field set.


In [ ]:
plot_group('linear', 'linear_apar_waltz')


### linear_apar_bpar_waltz

Generated controlled linear Waltz-style cases with both A_parallel and B_parallel enabled, at selected beta values. These isolate the coupled electromagnetic field-set behavior in a compact generated validation setup.


In [ ]:
plot_group('linear', 'linear_apar_bpar_waltz')


### linear_bpar_waltz

Generated controlled linear Waltz-style cases with B_parallel enabled without A_parallel. This tests the B_parallel-only path separately from the coupled A_parallel+B_parallel setting.


In [ ]:
plot_group('linear', 'linear_bpar_waltz')


### diag_lin_all_em

Adapted from `gkw_ref/tests/extra/diag_lin_all`. This is a broad linear diagnostic-regression setting: many diagnostic switches are enabled, plus collisions, `uprim`, `VCOR = 0.13`, and centrifugal flags (`cf_trap`, `cf_drift`). Current gyaradax does **not** support `VCOR`/rotation physics; use this section to inspect diagnostic/provenance behavior, not as a clean isolated EM benchmark.


In [ ]:
plot_group('linear', 'diag_lin_all_em')


## Nonlinear

Nonlinear EM validation cases grouped by tested setting. These are generated controlled Waltz-style nonlinear cases for comparing field-set behavior under ExB nonlinearity.


### nonlinear_apar_waltz

Generated nonlinear Waltz-style cases with A_parallel enabled and B_parallel disabled. These test nonlinear ExB evolution with the A_parallel electromagnetic field set.


In [ ]:
plot_group('nonlinear', 'nonlinear_apar_waltz')


### nonlinear_apar_bpar_waltz

Generated nonlinear Waltz-style cases with both A_parallel and B_parallel enabled. These test nonlinear ExB evolution with the coupled electromagnetic field set.


In [ ]:
plot_group('nonlinear', 'nonlinear_apar_bpar_waltz')


### nonlinear_bpar_only_waltz

Generated nonlinear Waltz-style cases with B_parallel enabled without A_parallel. These isolate the nonlinear B_parallel-only setting.


In [ ]:
plot_group('nonlinear', 'nonlinear_bpar_only_waltz')
